In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastbook import *

# Глубокое погружение в архитектуру приложений.


Сейчас мы находимся в уникальной ситуации, когда мы можем полностью понять архитектуры, которые мы использовали для создания наших передовых моделей в области компьютерного зрения, обработки естественного языка и анализа табличных данных. В этой главе мы подробно рассмотрим, как работают модели, используемые в fastai, и покажем, как создавать модели, аналогичные тем, что используются.

Мы также вернемся к конвейеру предварительной обработки данных, который мы видели в разделе <<chapter_midlevel_data>>, предназначенном для Siamese-сетей, и покажем, как можно использовать компоненты библиотеки fastai для создания собственных предварительно обученных моделей для новых задач.

Мы начнем с компьютерного зрения.

## Компьютерное зрение


Для приложений компьютерного зрения мы используем функции `vision_learner` и `unet_learner` для создания наших моделей, в зависимости от задачи. В этом разделе мы рассмотрим, как создавать объекты `Learner`, которые мы использовали в первых двух частях этой книги.

### vision_learner


Давайте рассмотрим, что происходит, когда мы используем функцию `vision_learner`. Сначала мы передаем этой функции архитектуру, которая будет использоваться для "тела" сети. В большинстве случаев мы используем ResNet, и, как вы уже знаете, как его создавать, поэтому мы не будем вдаваться в подробности. Предварительно обученные веса загружаются по мере необходимости и загружаются в ResNet.

Затем, для переноса обучения, сеть необходимо "разрезать". Это означает удаление последнего слоя, который отвечает только за категоризацию, специфичную для ImageNet. Фактически, мы удаляем не только этот слой, но и все, начиная с адаптивного слоя усреднения. Причина этого станет ясна через мгновение. Поскольку разные архитектуры могут использовать разные типы слоев усреднения или даже совершенно разные "головки", мы не просто ищем адаптивный слой усреднения, чтобы определить, где разрезать предварительно обученную модель. Вместо этого у нас есть словарь, содержащий информацию, которая используется для каждой модели, чтобы определить, где заканчивается ее "тело" и начинается ее "головка". Мы называем это `model_meta` – вот пример для resnet-50:


In [ ]:
model_meta[resnet50]

{'cut': -2,
 'split': <function fastai.vision.learner._resnet_split(m)>,
 'stats': ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])}

> терминология: Корпус и "головка": "Головка" нейронной сети – это часть, которая специализируется на выполнении определенной задачи. В случае сверточной нейронной сети (CNN), это обычно часть, расположенная после слоя адаптивного усредненного пулинга. "Корпус" включает в себя все остальное и включает в себя "ствол" (о котором мы узнали в разделе <<chapter_resnet>>).

Если мы возьмем все слои до точки отсечения `-2`, мы получим ту часть модели, которую библиотека fastai будет использовать для обучения с переносом знаний. Теперь мы добавляем новую "голову" (заголовок). Она создается с помощью функции `create_head`:


In [ ]:
#hide_output
create_head(20,2)

Sequential(
  (0): AdaptiveConcatPool2d(
    (ap): AdaptiveAvgPool2d(output_size=1)
    (mp): AdaptiveMaxPool2d(output_size=1)
  )
  (1): full: False
  (2): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.25, inplace=False)
  (4): Linear(in_features=20, out_features=512, bias=False)
  (5): ReLU(inplace=True)
  (6): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=512, out_features=2, bias=False)
)

```
Sequential(
  (0): AdaptiveConcatPool2d (
    (ap): AdaptiveAvgPool2d(output_size=1)
    (mp): AdaptiveMaxPool2d(output_size=1)
  )
  (1): Flatten()
  (2): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True)
  (3): Dropout(p=0.25, inplace=False)
  (4): Linear(in_features=20, out_features=512, bias=False)
  (5): ReLU(inplace=True)
  (6): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True)
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=512, out_features=2, bias=False)
)
```

С помощью этой функции вы можете указать, сколько дополнительных линейных слоев будет добавлено в конце, какой коэффициент dropout использовать после каждого из них, а также какой тип пулинга применять. По умолчанию, fastai применяет как усредненный, так и максимальный пулинг, а затем объединяет результаты (это слой `AdaptiveConcatPool2d`). Это не самый распространенный подход, но он был разработан независимо в fastai и других исследовательских лабораториях в последние годы, и, как правило, обеспечивает небольшое улучшение по сравнению с использованием только усредненного пулинга.

Fastai немного отличается от большинства библиотек тем, что по умолчанию добавляет два линейных слоя, а не один, в конце сверточной нейронной сети (CNN). Это связано с тем, что трансферное обучение может быть полезным даже в тех случаях, когда предварительно обученная модель используется в совершенно разных областях, как мы видели. Однако, использование только одного линейного слоя, скорее всего, будет недостаточно в этих случаях; мы обнаружили, что использование двух линейных слоев позволяет более быстро и легко применять трансферное обучение в большем количестве ситуаций.

> Примечание: Еще один слой Batch Normalization?: Один из параметров функции `create_head`, который стоит изучить, это `bn_final`. Установка этого параметра в значение `true` приведет к добавлению слоя Batch Normalization в качестве последнего слоя вашей модели. Это может быть полезно для того, чтобы ваша модель правильно масштабировала выходные данные. Мы пока не встречали публикаций, описывающих этот подход, но мы обнаружили, что он хорошо работает на практике, где бы мы его ни использовали.

Теперь давайте рассмотрим, что `unet_learner` делал в задаче сегментации, которую мы показали во введении к главе.

### unet_learner


Одна из наиболее интересных архитектур в области глубокого обучения — это та, которую мы использовали для сегментации, как было описано во введении к этой главе. Сегментация — это сложная задача, поскольку требуемый результат — это изображение или сетка пикселей, содержащая предсказанную метку для каждого пикселя. Существуют и другие задачи, имеющие схожую базовую структуру, такие как увеличение разрешения изображения (*суперразрешение*), добавление цвета к черно-белому изображению (*колоризация*) или преобразование фотографии в синтетическую картину (*перенос стиля*) — эти задачи рассматриваются в онлайн-главе этой книги, поэтому обязательно ознакомьтесь с ней после прочтения этой главы. В каждом случае мы начинаем с изображения и преобразуем его в другое изображение с теми же размерами или соотношением сторон, но с измененными пикселями. Мы называем эти модели *генеративными моделями компьютерного зрения*.

Мы решаем эту задачу, используя тот же подход к разработке "головы" сверточной нейронной сети (CNN), который мы видели в предыдущей задаче. Мы начинаем, например, с ResNet, затем удаляем слой адаптивного пулинга и все, что за ним следует. Затем мы заменяем эти слои нашей собственной "головой", которая выполняет генеративную задачу.

В предыдущем предложении было много упрощений! Как же нам создать "голову" CNN, которая генерирует изображение? Если мы начинаем, например, с входного изображения размером 224 пикселя, то в конце "тела" ResNet у нас будет сетка сверточных активаций размером 7x7. Как мы можем преобразовать это в маску сегментации размером 224 пикселя?

Конечно, мы делаем это с помощью нейронной сети! Поэтому нам нужен какой-то слой, который может увеличивать размер сетки в CNN. Один из самых простых подходов к этому — заменить каждый пиксель в сетке 7x7 на четыре пикселя в квадрате 2x2. Каждый из этих четырех пикселей будет иметь одинаковое значение — это называется *билинейной интерполяцией*. PyTorch предоставляет слой, который делает это за нас, поэтому один из вариантов — создать "голову", которая содержит сверточные слои с шагом 1 (наряду с обычными слоями нормализации пакетов и ReLU), чередующиеся со слоями билинейной интерполяции 2x2. На самом деле, вы можете попробовать это сейчас! Попробуйте создать собственную "голову" с такой структурой и примените ее к задаче сегментации CamVid. Вы должны получить некоторые разумные результаты, хотя они не будут такими хорошими, как результаты, представленные во введении к этой главе.

Другой подход — заменить комбинацию билинейной интерполяции и свертки на *транспонированную свертку*, также известную как *свертка с шагом 0,5*. Это идентично обычной свертке, но сначала между всеми пикселями входного сигнала добавляется заполнение нулями. Это легче всего увидеть на рисунке — <<transp_conv>> показывает схему из отличной статьи о сверточных вычислениях, которую мы обсуждали во введении к главе о свертках, показывающую транспонированную свертку 3x3, примененную к изображению 3x3.


```markdown
<img alt="Свертка с транспонированием" width="815" caption="Свертка с транспонированием (предоставлено Винсентом Дюмулином и Франческо Визином)" id="transp_conv" src="images/att_00051.png">
```

Как вы видите, результат этого заключается в увеличении размера входных данных. Вы можете попробовать это прямо сейчас, используя класс `ConvLayer` из библиотеки fastai; передайте параметр `transpose=True`, чтобы создать транспонированную свертку, а не обычную, в вашей собственной "голове" сети.

Однако ни один из этих подходов не работает идеально. Проблема в том, что наша сетка 7x7 просто не содержит достаточно информации для создания выходного изображения размером 224x224 пикселя. От активаций каждой ячейки сетки требуется огромное количество информации, чтобы полностью восстановить каждый пиксель в выходном изображении. Решением этой проблемы является использование *связей "перескока"*, как в ResNet, но при этом информация "перескакивает" от активаций основной части ResNet прямо к активациям транспонированной свертки, расположенной на противоположной стороне архитектуры. Этот подход, иллюстрированный в документе <<unet>>, был разработан Олафом Рённебергером, Филиппом Фишером и Томасом Броксом в статье 2015 года ["U-Net: Convolutional Networks for Biomedical Image Segmentation"](https://arxiv.org/abs/1505.04597). Хотя статья была посвящена медицинским приложениям, U-Net произвела революцию во всех видах генеративных моделей компьютерного зрения.

```markdown
<img alt="Архитектура U-Net" width="630" caption="Архитектура U-Net (предоставлено Олафом Роннебергером, Филиппом Фишером и Томасом Броксом)" id="unet" src="images/att_00052.png">
```

На этой картинке слева изображена структура CNN (в данном случае, это обычная CNN, а не ResNet, и используется максимальное объединение с окном 2x2 вместо сверток с шагом 2, поскольку эта статья была написана до появления ResNet), а справа – слои транспонированной свертки ("up-conv"). Затем показаны дополнительные "перекрестные" соединения, представленные серыми стрелками, идущими слева направо. Вы можете понять, почему это называется "U-Net"!

В этой архитектуре входные данные для слоев транспонированной свертки – это не только сетка с более низким разрешением в предыдущем слое, но и сетка с более высоким разрешением в блоке ResNet. Это позволяет U-Net использовать всю информацию из исходного изображения, когда это необходимо. Одной из сложностей U-Net является то, что точная архитектура зависит от размера изображения. В библиотеке fastai есть уникальный класс `DynamicUnet`, который автоматически генерирует архитектуру подходящего размера, основываясь на предоставленных данных.

Теперь давайте рассмотрим пример, в котором мы используем библиотеку fastai для создания собственной модели.

### Сиамская сеть


In [ ]:
#hide
from fastai.vision.all import *
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")

class SiameseImage(fastuple):
    def show(self, ctx=None, **kwargs): 
        img1,img2,same_breed = self
        if not isinstance(img1, Tensor):
            if img2.size != img1.size: img2 = img2.resize(img1.size)
            t1,t2 = tensor(img1),tensor(img2)
            t1,t2 = t1.permute(2,0,1),t2.permute(2,0,1)
        else: t1,t2 = img1,img2
        line = t1.new_zeros(t1.shape[0], t1.shape[1], 10)
        return show_image(torch.cat([t1,line,t2], dim=2), 
                          title=same_breed, ctx=ctx)
    
def label_func(fname):
    return re.match(r'^(.*)_\d+.jpg$', fname.name).groups()[0]

class SiameseTransform(Transform):
    def __init__(self, files, label_func, splits):
        self.labels = files.map(label_func).unique()
        self.lbl2files = {l: L(f for f in files if label_func(f) == l) for l in self.labels}
        self.label_func = label_func
        self.valid = {f: self._draw(f) for f in files[splits[1]]}
        
    def encodes(self, f):
        f2,t = self.valid.get(f, self._draw(f))
        img1,img2 = PILImage.create(f),PILImage.create(f2)
        return SiameseImage(img1, img2, t)
    
    def _draw(self, f):
        same = random.random() < 0.5
        cls = self.label_func(f)
        if not same: cls = random.choice(L(l for l in self.labels if l != cls)) 
        return random.choice(self.lbl2files[cls]),same
    
splits = RandomSplitter()(files)
tfm = SiameseTransform(files, label_func, splits)
tls = TfmdLists(files, tfm, splits=splits)
dls = tls.dataloaders(after_item=[Resize(224), ToTensor], 
    after_batch=[IntToFloatTensor, Normalize.from_stats(*imagenet_stats)])

Вернемся к конвейеру обработки данных, который мы настроили в главе <<chapter_midlevel_data>> для сети Siamese. Как вы помните, он состоял из пар изображений, причем метка была `True` или `False` в зависимости от того, принадлежали ли они к одному и тому же классу.

Используя то, что мы только что рассмотрели, давайте создадим собственную модель для этой задачи и обучим ее. Как? Мы будем использовать предварительно обученную архитектуру и пропускать через нее наши два изображения. Затем мы можем объединить результаты и передать их в специальный модуль, который вернет два предсказания. С точки зрения модулей, это выглядит следующим образом:


In [ ]:
class SiameseModel(Module):
    def __init__(self, encoder, head):
        self.encoder,self.head = encoder,head
    
    def forward(self, x1, x2):
        ftrs = torch.cat([self.encoder(x1), self.encoder(x2)], dim=1)
        return self.head(ftrs)

Для создания нашего энкодера нам нужно просто взять предварительно обученную модель и "обрезать" ее, как мы уже объясняли ранее. Функция `create_body` выполняет эту задачу за нас; нам нужно только указать, в каком месте мы хотим произвести "обрезание". Как мы видели ранее, согласно справочнику метаданных для предварительно обученных моделей, значение "обрезания" для модели ResNet равно `-2`:

In [ ]:
encoder = create_body(resnet34, cut=-2)

Затем мы можем создать "голову" (head) нашей модели. Анализ энкодера показывает, что последний слой содержит 512 признаков, поэтому эта "голова" должна получать `512*2`. Почему 2? Мы должны умножить на 2, потому что у нас есть два изображения. Таким образом, мы создаем "голову" следующим образом:


In [ ]:
head = create_head(512*2, 2, ps=0.5)

С использованием нашего кодировщика и декодировщика мы можем теперь построить нашу модель:


In [ ]:
model = SiameseModel(encoder, head)

Перед использованием `Learner`, нам нужно определить еще две вещи. Во-первых, необходимо определить функцию потерь, которую мы хотим использовать. Это стандартная кросс-энтропия, но поскольку наши целевые значения являются булевыми, нам нужно преобразовать их в целые числа, иначе PyTorch выдаст ошибку:


In [ ]:
def loss_func(out, targ):
    return nn.CrossEntropyLoss()(out, targ.long())

Более важно то, что для максимального использования обучения с переносом, нам необходимо определить собственный *разделитель*. Разделитель – это функция, которая сообщает библиотеке fastai, как разделить модель на группы параметров. Эти группы используются в фоновом режиме для обучения только "головы" модели при использовании обучения с переносом.

В данном случае нам требуются две группы параметров: одна для кодировщика и одна для "головы". Таким образом, мы можем определить следующий разделитель (функция `params` просто возвращает все параметры заданного модуля):

In [ ]:
def siamese_splitter(model):
    return [params(model.encoder), params(model.head)]

Затем мы можем определить наш объект `Learner`, передав ему данные, модель, функцию потерь, инструмент для разделения данных и любые метрики, которые мы хотим использовать. Поскольку мы не используем удобную функцию из библиотеки fastai для переноса обучения (например, `vision_learner`), нам нужно вручную вызвать метод `learn.freeze`. Это обеспечит, что будет обучена только последняя группа параметров (в данном случае, "головная" часть модели).

In [ ]:
learn = Learner(dls, model, loss_func=loss_func, 
                splitter=siamese_splitter, metrics=accuracy)
learn.freeze()

Затем мы можем обучить нашу модель, используя обычные методы:

In [ ]:
learn.fit_one_cycle(4, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,0.367015,0.281242,0.885656,00:26
1,0.307688,0.214721,0.915426,00:26
2,0.275221,0.170615,0.936401,00:26
3,0.223771,0.159633,0.943843,00:26


Перед тем, как полностью "разморозить" модель и немного ее доработать, используя различные скорости обучения (то есть: более низкую скорость обучения для основной части модели и более высокую для ее "головы"):

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(4, slice(1e-6,1e-4))

epoch,train_loss,valid_loss,accuracy,time
0,0.212744,0.159033,0.944520,00:35
1,0.201893,0.159615,0.942490,00:35
2,0.204606,0.152338,0.945196,00:36
3,0.213203,0.148346,0.947903,00:36


94,8% – это очень хороший результат, особенно если учесть, что у классификатора, обученного таким же образом (без использования методов увеличения объема данных), показатель точности составлял всего 7%.

Теперь, когда мы рассмотрели, как создавать современные и передовые модели компьютерного зрения, давайте перейдем к обработке естественного языка (NLP).

## Обработка естественного языка


Преобразование языковой модели AWD-LSTM в классификатор, использующий обучение с переносом, как мы сделали в разделе <<chapter_nlp>>, проходит по очень схожему процессу, чем то, что мы использовали с `vision_learner` в первой части этого раздела. В данном случае нам не нужен "мета"-словарь, поскольку мы не поддерживаем такое разнообразие архитектур в основной части кода. Нам нужно лишь выбрать стековую рекуррентную нейронную сеть (RNN) для кодировщика в языковой модели, что является одним модулем PyTorch. Этот кодировщик будет предоставлять активации для каждого слова входных данных, поскольку языковая модель должна выдавать прогноз для каждого следующего слова.

Для создания классификатора мы используем подход, описанный в статье [ULMFiT](https://arxiv.org/abs/1801.06146) как "BPTT для классификации текста (BPT3C)":


Вот перевод:

> Мы разделяем документ на пакеты фиксированной длины размером *b*. В начале каждого пакета модель инициализируется с использованием конечного состояния предыдущего пакета; мы отслеживаем скрытые состояния для усреднения и максимизации при агрегации; градиенты распространяются обратно к тем пакетам, скрытые состояния которых внесли вклад в окончательное предсказание. На практике мы используем последовательности обратного распространения переменной длины.

Другими словами, классификатор содержит цикл `for`, который проходит по каждой порции последовательности. Состояние сохраняется между порциями, и активации каждой порции сохраняются. В конце мы используем тот же прием усреднения и максимального объединения, который мы используем для моделей компьютерного зрения, но в этот раз мы не объединяем данные по ячейкам сверточной нейронной сети (CNN), а по последовательностям рекуррентной нейронной сети (RNN).

Для этого цикла `for` нам нужно собирать данные в порциях, но каждый текстовый фрагмент должен обрабатываться отдельно, поскольку у каждого из них свои метки. Однако, весьма вероятно, что эти текстовые фрагменты будут иметь разную длину, что означает, что мы не сможем поместить их все в один массив, как мы делали с языковой моделью.

Именно здесь на помощь приходит заполнение (padding): когда мы собираем группу текстовых фрагментов, мы определяем самый длинный из них, а затем заполняем более короткие фрагменты специальным токеном под названием `xxpad`. Чтобы избежать крайних случаев, когда у нас есть текстовый фрагмент с 2000 токенами в той же порции, что и фрагмент с 10 токенами (что приведет к большому объему заполнения и значительным вычислительным затратам), мы корректируем случайность, чтобы текстовые фрагменты сравнимой длины были объединены вместе. Текстовые фрагменты все равно будут находиться в несколько случайном порядке для обучающего набора (для проверочного набора мы можем просто отсортировать их по длине), но не в полностью случайном порядке.

Все это делается автоматически в фоновом режиме библиотекой fastai при создании наших объектов `DataLoaders`.

## Табличная форма.


Наконец, давайте рассмотрим модели `fastai.tabular`. (Нам не нужно рассматривать коллаборативную фильтрацию отдельно, поскольку мы уже видели, что эти модели – это просто табличные модели, или они используют подход, основанный на скалярном произведении, который мы реализовали ранее с нуля).

Вот метод `forward` для класса `TabularModel`:

```python
if self.n_emb != 0:
    x = [e(x_cat[:,i]) for i,e in enumerate(self.embeds)]
    x = torch.cat(x, 1)
    x = self.emb_drop(x)
if self.n_cont != 0:
    x_cont = self.bn_cont(x_cont)
    x = torch.cat([x, x_cont], 1) if self.n_emb != 0 else x_cont
return self.layers(x)
```

Мы не будем показывать метод `__init__`, так как он не представляет особого интереса, но мы рассмотрим каждую строку кода в методе `forward` по отдельности. Первая строка:


```python
if self.n_emb != 0:
```
Это просто проверка, есть ли какие-либо векторы эмбеддингов, с которыми нужно работать. Если у нас только непрерывные переменные, можно пропустить эту часть. `self.embeds` содержит матрицы эмбеддингов, поэтому здесь извлекаются активации для каждой из них:

```python
    x = [e(x_cat[:,i]) for i,e in enumerate(self.embeds)]
```

и они объединяются в один тензор:

```python
    x = torch.cat(x, 1)
```

Затем применяется слой dropout. Вы можете передать значение `embd_p` в функцию `__init__`, чтобы изменить это значение:

```python
    x = self.emb_drop(x)
```

Теперь мы проверяем, есть ли какие-либо непрерывные переменные, с которыми нужно работать:

```python
if self.n_cont != 0:
```

Они проходят через слой пакетной нормализации:

```python
    x_cont = self.bn_cont(x_cont)
```

и объединяются с активациями эмбеддингов, если они есть:

```python
    x = torch.cat([x, x_cont], 1) if self.n_emb != 0 else x_cont
```

Наконец, этот тензор передается через линейные слои (каждый из которых включает пакетную нормализацию, если `use_bn` имеет значение `True`, и dropout, если `ps` установлено в какое-либо значение или список значений):

```python
return self.layers(x)

```

Поздравляем! Теперь вы знаете каждую деталь архитектур, используемых в библиотеке fastai!


## Заключение о архитектурных подходах.

Как вы видите, детали архитектур глубокого обучения сейчас не должны вас пугать. Вы можете изучить код библиотек fastai и PyTorch и понять, что именно там происходит. Гораздо важнее попытаться понять, *почему* это происходит. Ознакомьтесь со статьями, на которые ссылаются в коде, и постарайтесь понять, как код соответствует описанным алгоритмам.

Теперь, когда мы рассмотрели все компоненты модели и данные, которые в нее поступают, мы можем подумать о том, что это значит для практического применения глубокого обучения. Если у вас неограниченные данные, неограниченный объем памяти и неограниченное время, то решение простое: обучите огромную модель на всех ваших данных в течение очень длительного времени. Но именно поэтому глубокое обучение не всегда является простым процессом: ваши данные, память и время обычно ограничены. Если у вас заканчивается память или время, то решение – обучить модель меньшего размера. Если вы не можете обучать модель достаточно долго, чтобы добиться переобучения, то вы не используете все возможности вашей модели.

Итак, первый шаг – это достичь состояния, когда можно добиться переобучения. Затем возникает вопрос о том, как уменьшить это переобучение. Раздел <<reduce_overfit>> показывает, как мы рекомендуем расставлять приоритеты в дальнейших шагах.

```markdown
<img alt="Шаги по снижению переобучения" width="400" caption="Шаги по снижению переобучения" id="reduce_overfit" src="images/att_00047.png">
```

Многие специалисты, столкнувшись с моделью, склонной к переобучению, начинают с совершенно не того конца этого процесса. Их первым шагом является использование более простой модели или увеличение регуляризации. Использование более простой модели должно быть абсолютно последним шагом, если только обучение вашей модели не занимает слишком много времени или требует слишком много памяти. Уменьшение размера вашей модели снижает ее способность выявлять тонкие взаимосвязи в ваших данных.

Вместо этого, ваш первый шаг должен заключаться в том, чтобы *собрать больше данных*. Это может включать в себя добавление большего количества меток к существующим данным, поиск дополнительных задач, которые ваша модель могла бы решать (или, другими словами, определение различных типов меток, которые можно использовать для обучения модели), или создание дополнительных синтетических данных с использованием различных методов расширения данных. Благодаря развитию таких подходов, как Mixup и подобных, эффективное расширение данных теперь доступно практически для всех типов данных.

Как только вы соберете максимально возможное количество данных и будете использовать их наиболее эффективно, используя все доступные метки и применяя все разумные методы расширения данных, если вы все еще сталкиваетесь с переобучением, вам следует рассмотреть возможность использования более обобщенных архитектур. Например, добавление пакетной нормализации может улучшить обобщающую способность модели.

Если вы все еще сталкиваетесь с переобучением, несмотря на все ваши усилия по использованию данных и настройке архитектуры, тогда вы можете обратить внимание на регуляризацию. В целом, добавление dropout к последнему или двум последним слоям хорошо зарегулирует вашу модель. Однако, как мы узнали из истории разработки AWD-LSTM, часто оказывается, что добавление dropout различных типов по всей модели может быть еще более эффективным. В целом, более крупная модель с большей регуляризацией более гибкая и, следовательно, может быть более точной, чем меньшая модель с меньшей регуляризацией.

Только после рассмотрения всех этих вариантов мы рекомендуем вам попробовать использовать более простую версию вашей архитектуры.

## Анкета


1. Что такое "головная часть" нейронной сети?
2. Что такое "тело" нейронной сети?
3. Что означает "усечение" нейронной сети? Зачем это нужно для обучения с переносом (transfer learning)?
4. Что такое `model_meta`? Попробуйте вывести его на экран, чтобы увидеть, что в нем содержится.
5. Прочитайте исходный код функции `create_head` и убедитесь, что вы понимаете, что делает каждая строка.
6. Посмотрите на вывод функции `create_head` и убедитесь, что вы понимаете, зачем нужны каждый слой и как исходный код `create_head` его создал.
7. Найдите способ изменить параметры dropout, размер слоя и количество слоев, создаваемых `vision_learner`, и попробуйте найти значения, которые приводят к повышению точности распознавания изображений животных.
8. Что делает `AdaptiveConcatPool2d`?
9. Что такое "интерполяция ближайшего соседа"? Как ее можно использовать для увеличения разрешения сверточных признаков?
10. Что такое "транспонированная свертка"? Как ее еще называют?
11. Создайте сверточный слой с параметром `transpose=True` и примените его к изображению. Проверьте форму выходных данных.
12. Нарисуйте архитектуру U-Net.
13. Что такое "Обратное распространение ошибки для классификации текста" (BPT3C)?
14. Как мы обрабатываем последовательности разной длины в BPT3C?
15. Попробуйте запустить каждую строку функции `TabularModel.forward` отдельно, по одной строке в ячейке, в блокноте, и посмотрите на формы входных и выходных данных на каждом шаге.
16. Как определен атрибут `self.layers` в классе `TabularModel`?
17. Какие пять шагов можно предпринять для предотвращения переобучения?
18. Почему мы не уменьшаем сложность архитектуры, прежде чем пробовать другие подходы к предотвращению переобучения?

### Дальнейшие исследования


1. Напишите свой собственный пользовательский модуль (head) и попробуйте обучить систему распознавания животных с его использованием. Посмотрите, сможете ли вы получить лучший результат, чем у стандартного модуля fastai.
2. Попробуйте поочередно использовать `AdaptiveConcatPool2d` и `AdaptiveAvgPool2d` в пользовательском модуле сверточной нейронной сети (CNN) и посмотрите, какую разницу это вносит.
3. Напишите свой собственный пользовательский разделитель данных, который будет создавать отдельные группы параметров для каждого блока ResNet и отдельную группу для начального слоя (stem). Попробуйте обучить систему с его использованием и посмотрите, улучшит ли это систему распознавания животных.
4. Прочитайте онлайн-главу о генеративных моделях изображений и создайте свой собственный инструмент для цветокоррекции, модели повышения разрешения или модели переноса стилей.
5. Создайте пользовательский модуль, использующий интерполяцию ближайшего соседа, и используйте его для сегментации изображений в наборе данных CamVid.